In [ ]:
!pip install kaggle==1.5.16

In [ ]:
! chmod 600 .kaggle/kaggle.json

In [ ]:
! kaggle competitions download Walmart-Recruiting-Store-Sales-Forecasting

In [ ]:
! unzip Walmart-Recruiting-Store-Sales-Forecasting.zip

In [ ]:
! unzip features.csv.zip
! unzip test.csv.zip
! unzip train.csv.zip

# TimesFM — Walmart Store Sales Forecasting (Bonus)

**Model:** Google's TimesFM — a pretrained time-series foundation model (decoder-only
Transformer, patch-based, trained on 100M+ real-world series).

**Two modes tested:**
1. **Zero-shot** — no training at all, straight inference using Google's pretrained weights
2. **Fine-tuned** — the pretrained model is further trained on Walmart's own sales history

**Note:** TimesFM's API has changed across versions and is under active development.
If a cell errors due to an API mismatch, check the official quickstart at
https://github.com/google-research/timesfm for the current signature — the core
logic here (load → zero-shot forecast → fine-tune → forecast again) should remain valid
even if exact method names shift.

In [ ]:
!pip install "numpy<2" "torch==2.2.0" timesfm[torch] mlflow dagshub wandb -q --force-reinstall

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, os
import torch
import wandb
import mlflow
import mlflow.pyfunc

import timesfm

WANDB_ENTITY  = 'ikvas22-free-university-of-tbilisi'
WANDB_PROJECT = 'Walmart Weekly Sales Prediction'
WANDB_GROUP   = 'TimesFM'

MLFLOW_EXPERIMENT = 'TimesFM_Training'
MLFLOW_MODEL_NAME  = 'timesfm_walmart_best'

TRAIN_PATH    = 'train.csv'
FEATURES_PATH = 'features.csv'
STORES_PATH   = 'stores.csv'

H          = 4     # forecast horizon (weeks) — same as other 3 models
INPUT_SIZE = 52     # context length given to TimesFM
VAL_START  = '2012-04-01'
N_WINDOWS  = 7       # same rolling windows as evaluate_cv in other notebooks

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

wandb.login()

print(f'Setup OK — device: {DEVICE}')

## 1. Pre-processing

In [ ]:
run = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'preprocessing',
    name     = 'TimesFM_Preprocessing',
)

train_raw    = pd.read_csv(TRAIN_PATH,    parse_dates=['Date'])
features_raw = pd.read_csv(FEATURES_PATH, parse_dates=['Date'])
stores_raw   = pd.read_csv(STORES_PATH)

df = (
    train_raw
    .merge(features_raw, on=['Store', 'Date', 'IsHoliday'], how='left')
    .merge(stores_raw,   on=['Store'],                       how='left')
)

wandb.log({
    'raw_rows' : df.shape[0],
    'raw_cols' : df.shape[1],
    'stores'   : df['Store'].nunique(),
    'depts'    : df['Dept'].nunique(),
    'date_min' : str(df['Date'].min().date()),
    'date_max' : str(df['Date'].max().date()),
})

# TimesFM (like N-BEATS/DLinear) only needs the raw sales series —
# it's a univariate foundation model, no engineered features consumed.
df['unique_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
df_nf = (
    df[['unique_id', 'Date', 'Weekly_Sales', 'IsHoliday']]
    .rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

min_len    = INPUT_SIZE + H
series_len = df_nf.groupby('unique_id')['ds'].count()
valid_ids  = series_len[series_len >= min_len].index
dropped    = series_len[series_len < min_len].index.tolist()
df_nf      = df_nf[df_nf['unique_id'].isin(valid_ids)].reset_index(drop=True)

wandb.log({
    'total_series'   : len(series_len),
    'valid_series'   : len(valid_ids),
    'dropped_series' : len(dropped),
})

print(f'Valid series : {len(valid_ids)}')
print(f'Dropped      : {len(dropped)} (shorter than {min_len} weeks)')

df_train = df_nf[df_nf['ds'] <  VAL_START].copy()
df_val   = df_nf[df_nf['ds'] >= VAL_START].copy()
df_full  = pd.concat([df_train, df_val]).sort_values(['unique_id', 'ds']).reset_index(drop=True)

wandb.log({
    'train_rows'     : len(df_train),
    'val_rows'       : len(df_val),
    'val_start'      : VAL_START,
    'horizon_weeks'  : H,
    'context_weeks'  : INPUT_SIZE,
})

print(f'Train : {df_train["ds"].min().date()} → {df_train["ds"].max().date()}  ({len(df_train):,} rows)')
print(f'Val   : {df_val["ds"].min().date()} → {df_val["ds"].max().date()}  ({len(df_val):,} rows)')

wandb.finish()

## 2. Training

Since TimesFM is not a `neuralforecast` model, we can't use `NeuralForecast.cross_validation()`
directly. Instead we replicate the same rolling-window logic manually — same `N_WINDOWS=7`,
same `step_size=H` — so the WMAE is directly comparable to N-BEATS / DLinear / TFT.

In [ ]:
def wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray) -> float:
    weights = np.where(is_holiday, 5.0, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


def rolling_windows(df_series: pd.DataFrame, n_windows: int, h: int, input_size: int):
    """
    Yields (context_dates, target_dates) pairs for each rolling window,
    mirroring neuralforecast's cross_validation(n_windows, step_size=h).
    Windows walk backward from the end of df_series in steps of h.
    """
    dates = sorted(df_series['ds'].unique())
    for w in range(n_windows, 0, -1):
        cutoff_idx = len(dates) - w * h
        if cutoff_idx < input_size:
            continue
        context_dates = dates[max(0, cutoff_idx - input_size):cutoff_idx]
        target_dates  = dates[cutoff_idx:cutoff_idx + h]
        yield context_dates, target_dates


def evaluate_timesfm(tfm_model, df_source: pd.DataFrame, n_windows: int = N_WINDOWS) -> tuple:
    """
    Rolling-window zero-shot / fine-tuned evaluation.
    For each series and each window: feed INPUT_SIZE weeks of context,
    forecast H weeks ahead, compare against actual.
    """
    all_preds  = []
    unique_ids = df_source['unique_id'].unique()

    for uid in unique_ids:
        s = df_source[df_source['unique_id'] == uid].sort_values('ds')
        for context_dates, target_dates in rolling_windows(s, n_windows, H, INPUT_SIZE):
            context = s[s['ds'].isin(context_dates)]['y'].values
            target  = s[s['ds'].isin(target_dates)]
            if len(context) < 1 or len(target) == 0:
                continue

            point_forecast, _ = tfm_model.forecast(
                [context], freq=[0]     # freq=0 → auto / weekly-scale
            )
            pred = point_forecast[0][:len(target)]

            for i, (_, row) in enumerate(target.iterrows()):
                all_preds.append({
                    'unique_id' : uid,
                    'ds'        : row['ds'],
                    'y'         : row['y'],
                    'y_pred'    : pred[i] if i < len(pred) else np.nan,
                    'IsHoliday' : row['IsHoliday'],
                })

    eval_df = pd.DataFrame(all_preds).dropna(subset=['y_pred'])
    score_wmae = wmae(eval_df['y'].values, eval_df['y_pred'].values, eval_df['IsHoliday'].values)
    score_mae  = float(np.mean(np.abs(eval_df['y'].values - eval_df['y_pred'].values)))
    return score_wmae, score_mae, eval_df

In [ ]:
# ── Zero-shot: load pretrained TimesFM, no training ────────────
# Uses Google's pretrained checkpoint straight off the shelf.
run_zeroshot = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'training',
    name     = 'TimesFM_ZeroShot',
    config   = {
        'checkpoint'  : 'google/timesfm-2.0-500m-pytorch',
        'context_len' : INPUT_SIZE,
        'horizon'     : H,
        'mode'        : 'zero-shot',
    }
)

tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend             = DEVICE,
        per_core_batch_size = 32,
        horizon_len         = H,
        context_len         = INPUT_SIZE,
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id='google/timesfm-2.0-500m-pytorch'
    ),
)

zeroshot_wmae, zeroshot_mae, eval_zeroshot = evaluate_timesfm(tfm, df_full, n_windows=N_WINDOWS)

wandb.log({
    'val_wmae' : zeroshot_wmae,
    'val_mae'  : zeroshot_mae,
})

print(f'Zero-shot WMAE : {zeroshot_wmae:,.2f}')
print(f'Zero-shot MAE  : {zeroshot_mae:,.2f}')

wandb.finish()

In [ ]:
# ── Fine-tuning: continue training the pretrained checkpoint ──
# on Walmart's own training history. Uses a small learning rate
# since we're adapting an already-converged foundation model,
# not training from scratch.
#
# NOTE: TimesFM's fine-tuning API has changed across releases.
# If `tfm.finetune(...)` doesn't exist in your installed version,
# check https://github.com/google-research/timesfm/tree/master/experiments
# for the current finetuning example — the surrounding pipeline
# (data prep, evaluate_timesfm, MLflow registry) stays the same
# regardless of the exact fine-tuning call.
run_finetune = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'training',
    name     = 'TimesFM_FineTune',
    config   = {
        'checkpoint'    : 'google/timesfm-2.0-500m-pytorch',
        'context_len'   : INPUT_SIZE,
        'horizon'       : H,
        'mode'          : 'fine-tuned',
        'learning_rate' : 1e-5,
        'epochs'        : 3,
        'batch_size'    : 32,
    }
)

# Build (context, target) training pairs from df_train using the
# same sliding-window logic as evaluation, so train/eval are consistent.
train_examples = []
for uid in df_train['unique_id'].unique():
    s = df_train[df_train['unique_id'] == uid].sort_values('ds')['y'].values
    for start in range(0, len(s) - INPUT_SIZE - H, H):
        context = s[start : start + INPUT_SIZE]
        target  = s[start + INPUT_SIZE : start + INPUT_SIZE + H]
        if len(context) == INPUT_SIZE and len(target) == H:
            train_examples.append((context, target))

print(f'Built {len(train_examples):,} fine-tuning examples')
wandb.log({'finetune_examples': len(train_examples)})

try:
    # Preferred path: TimesFM's built-in finetune method (if available
    # in your installed version)
    tfm.finetune(
        train_examples,
        learning_rate = 1e-5,
        epochs        = 3,
        batch_size    = 32,
    )
    print('Fine-tuning complete via tfm.finetune()')
except AttributeError:
    print('tfm.finetune() not found in this version of the library.')
    print('See https://github.com/google-research/timesfm for the current')
    print('finetuning example matching your installed version.')
    print('Falling back to the zero-shot model for the rest of this notebook.')

finetune_wmae, finetune_mae, eval_finetune = evaluate_timesfm(tfm, df_full, n_windows=N_WINDOWS)

wandb.log({
    'val_wmae'          : finetune_wmae,
    'val_mae'           : finetune_mae,
    'zeroshot_wmae'     : zeroshot_wmae,
    'wmae_improvement'  : zeroshot_wmae - finetune_wmae,
})

print(f'Fine-tuned WMAE : {finetune_wmae:,.2f}')
print(f'Zero-shot WMAE  : {zeroshot_wmae:,.2f}')
print(f'Improvement     : {zeroshot_wmae - finetune_wmae:,.2f}')

# Use whichever performed better for the registry
if finetune_wmae < zeroshot_wmae:
    best_wmae, best_mae, eval_best = finetune_wmae, finetune_mae, eval_finetune
    best_mode = 'fine-tuned'
else:
    best_wmae, best_mae, eval_best = zeroshot_wmae, zeroshot_mae, eval_zeroshot
    best_mode = 'zero-shot'

print(f'\nBest mode: {best_mode}  (WMAE {best_wmae:,.2f})')

# Prediction plots
sample_ids = eval_best['unique_id'].unique()[:6]
fig, axes  = plt.subplots(3, 2, figsize=(14, 10))
axes       = axes.flatten()

for ax, uid in zip(axes, sample_ids):
    history = df_train[df_train['unique_id'] == uid].tail(52)
    actual  = eval_best[eval_best['unique_id'] == uid]
    ax.plot(history['ds'], history['y'],     label='History', color='steelblue')
    ax.plot(actual['ds'],  actual['y'],      label='Actual',  color='black')
    ax.plot(actual['ds'],  actual['y_pred'], label='TimesFM', color='tomato', linestyle='--')
    hol = actual[actual['IsHoliday'] == 1]
    ax.scatter(hol['ds'], hol['y'], color='gold', zorder=5, s=40, label='Holiday')
    ax.set_title(f'Store-Dept {uid}', fontsize=10)
    ax.legend(fontsize=7)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle(f'TimesFM ({best_mode}) — Validation Predictions', fontsize=13)
plt.tight_layout()
wandb.log({'prediction_plots': wandb.Image(fig)})
plt.show()

per_series = (
    eval_best
    .groupby('unique_id')
    .apply(lambda g: wmae(g['y'].values, g['y_pred'].values, g['IsHoliday'].values))
    .reset_index()
    .rename(columns={0: 'wmae'})
    .sort_values('wmae', ascending=False)
)
wandb.log({'per_series_wmae': wandb.Table(dataframe=per_series)})

wandb.finish()

## 3. Save Best Model to MLflow Registry

In [ ]:
import dagshub
dagshub.init(repo_owner='ikvas22', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)

In [ ]:
class TimesFMWrapper(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        with open(context.artifacts['tfm_model'], 'rb') as f:
            self.tfm = pickle.load(f)
        self.input_size = 52
        self.h          = 4

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        """
        Accepts raw DataFrame with [Store, Dept, Date, Weekly_Sales].
        Returns [Store, Dept, Date, Weekly_Sales_pred] for the next
        H weeks after each series' last observed date.
        """
        df_in = model_input.copy()
        df_in['Date']      = pd.to_datetime(df_in['Date'])
        df_in['unique_id'] = df_in['Store'].astype(str) + '_' + df_in['Dept'].astype(str)

        results = []
        for uid, group in df_in.groupby('unique_id'):
            g = group.sort_values('Date')
            context_vals = g['Weekly_Sales'].values[-self.input_size:]
            if len(context_vals) == 0:
                continue

            point_forecast, _ = self.tfm.forecast([context_vals], freq=[0])
            pred = point_forecast[0][:self.h]

            last_date = g['Date'].max()
            future_dates = pd.date_range(
                start=last_date + pd.Timedelta(weeks=1), periods=self.h, freq='W-FRI'
            )
            store, dept = uid.split('_')
            for d, p in zip(future_dates, pred):
                results.append({
                    'Store': int(store), 'Dept': int(dept),
                    'Date': d, 'Weekly_Sales_pred': p
                })

        return pd.DataFrame(results)


os.makedirs('mlflow_artifacts', exist_ok=True)
model_path = 'mlflow_artifacts/timesfm_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(tfm, f)

with mlflow.start_run(run_name='TimesFM_Best_Model') as run:
    mlflow.log_params({
        'mode'        : best_mode,
        'context_len' : INPUT_SIZE,
        'horizon'     : H,
        'checkpoint'  : 'google/timesfm-2.0-500m-pytorch',
    })
    mlflow.log_metrics({
        'val_wmae'      : best_wmae,
        'val_mae'       : best_mae,
        'zeroshot_wmae' : zeroshot_wmae,
    })
    mlflow.pyfunc.log_model(
        artifact_path         = 'timesfm_model',
        python_model          = TimesFMWrapper(),
        artifacts             = {'tfm_model': model_path},
        registered_model_name = MLFLOW_MODEL_NAME,
    )
    print(f'Registered: {MLFLOW_MODEL_NAME}')
    print(f'Run ID    : {run.info.run_id}')
    print(f'Best mode : {best_mode}')
    print(f'Best WMAE : {best_wmae:,.2f}')

In [ ]:
loaded = mlflow.pyfunc.load_model(f'models:/{MLFLOW_MODEL_NAME}/latest')
sample = train_raw[train_raw['Store'] == 1][['Store', 'Dept', 'Date', 'Weekly_Sales']].head(100)
result = loaded.predict(sample)
print('Registry load OK')
print(result.head())